In [1]:
import os
import sys
import time
import dask
import zarr
import xesmf as xe
import numpy as np
import xarray as xr
from glob import glob

In [2]:
import pandas as pd
from tqdm import tqdm

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
from scipy.stats import norm

In [5]:
from scipy.optimize import minimize
from scipy.special import gamma as gamma_func, gammainc, gammaincc, beta as beta_func
from scipy.stats import gamma as gamma_dist
from tqdm import tqdm

In [6]:
from scipy.special import gammaln

### Get post-processed fcsts

In [8]:
VAR = VAR_CESM = 'PRECT'
VAR_ERA5 = 'total_precipitation'
unit_CESM = 60 * 60 * 24 * 1000 # m/s to mm/day
unit_ERA5 = 1000  # m/day to mm/day
EPS = 1e-6        # floor for predicted variance
verif_years = np.arange(2010, 2011)

In [9]:
lat_ind = np.arange(192)

list_coef = []
for lat_i in lat_ind:
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/EMOS/{VAR}/emos_coef_lat_ind_{lat_i}.zarr'
    list_coef.append(xr.open_zarr(fn))

In [10]:
ds_emos = xr.concat(list_coef, dim='lat')
ds_emos = ds_emos.load()
ds_emos = ds_emos.transpose('lead_time', 'lat', 'lon')

In [11]:
list_input = []
for year in verif_years:
    fn_CESM = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year}-11-01_daily_ensemble.zarr'
    ds_CESM = xr.open_zarr(fn_CESM)[[VAR,]].sel(time=slice(f"{year+1}-01-01", f"{year+10}-12-31"))
    ds_CESM = ds_CESM.rename({'time': 'lead_time'})
    ds_CESM['lead_time'] = np.arange(3650) # 10 non-leap year, 365 day on each 
    list_input.append(ds_CESM)

ds_input = xr.concat(list_input, dim='init_time')
ds_input = ds_input.assign_coords({'init_time': verif_years+1})

list_target = []
for year in range(2009, 2026):
    fn_ERA5 = f'/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/ERA5_{year}.zarr'
    ds_ERA5 = xr.open_zarr(fn_ERA5)[[VAR_ERA5,]].rename({VAR_ERA5: VAR})    
    list_target.append(ds_ERA5)

ds_target = xr.concat(list_target, dim='time')

ds_input[VAR_CESM] = ds_input[VAR_CESM] * unit_CESM
ds_target[VAR_CESM] = ds_target[VAR_CESM] * unit_ERA5

In [12]:
# ════════════════════════════════════════════════════════════════════════
# Step 1: Noleap calendar → real-date index mapping
# ════════════════════════════════════════════════════════════════════════
 
def build_date_indices(init_years, n_lead, target_times):
    """
    Map every (init_time, lead_time) pair to an integer index into the
    target (ERA5) time axis.  Returns -1 where no matching date exists.
    
    The CESM2 noleap calendar has exactly 365 days per year (no Feb 29).
    For each model day we find the corresponding real-calendar date,
    then look it up in the ERA5 time coordinate.
    """
    time_to_idx = {pd.Timestamp(t): i for i, t in enumerate(target_times)}
    indices = np.full((len(init_years), n_lead), -1, dtype=np.int32)
 
    for i, year in enumerate(init_years):
        year = int(year)
        # Build the 3650-day noleap calendar for this initialisation
        noleap_dates = []
        for y in range(year, year + 10):
            days = pd.date_range(f'{y}-01-01', f'{y}-12-31', freq='D')
            days = days[~((days.month == 2) & (days.day == 29))]  # drop leap day
            noleap_dates.extend(days.tolist())
 
        for L in range(min(n_lead, len(noleap_dates))):
            d = noleap_dates[L]
            if d in time_to_idx:
                indices[i, L] = time_to_idx[d]
 
    return indices
 
 
print("Building noleap → real-calendar date mapping …")
init_years = ds_input.init_time.values           # [1959, 1960, …, 2010]
n_init     = len(init_years)
n_lead     = len(ds_input.lead_time)              # 3650
n_lat      = len(ds_input.lat)
n_lon      = len(ds_input.lon)
 
target_times_pd = pd.DatetimeIndex(ds_target.time.values)
date_idx = build_date_indices(init_years, n_lead, target_times_pd)
# date_idx shape: (n_init, n_lead) — value = position in target time axis
 
# Quick sanity check
n_valid_total = (date_idx >= 0).sum()
print(f"  Matched {n_valid_total} of {n_init * n_lead} (init, lead) pairs to ERA5 dates.\n")

Building noleap → real-calendar date mapping …
  Matched 3650 of 3650 (init, lead) pairs to ERA5 dates.



In [13]:
 
def csgd_params_from_moments(mu, sigma, delta):
    """
    Convert (mu, sigma, delta) to Gamma (shape, scale).
    mu must be > delta for valid parameters.
    """
    mu_shifted = np.maximum(mu - delta, EPS)
    k     = (mu_shifted / np.maximum(sigma, EPS)) ** 2      # shape
    theta = np.maximum(sigma, EPS) ** 2 / mu_shifted         # scale
    return k, theta


def apply_emos_csgd(ds_input, ds_emos, var=VAR):
    """
    Apply fitted CSGD-EMOS to produce calibrated forecast parameters.
 
    Returns xr.Dataset with:
        emos_k     : Gamma shape         (init_time, lead_time, lat, lon)
        emos_theta : Gamma scale          "
        emos_delta : shift                "
        emos_mu    : predictive mean      "
        emos_prob0 : P(precip = 0)        "
    """
    ens_mean = ds_input[var].mean(dim='member')
    ens_std  = ds_input[var].std(dim='member', ddof=1)
    ens_mean = np.maximum(ens_mean, 0.0)
    ens_std  = np.maximum(ens_std, EPS)
 
    a0    = ds_emos['a0']
    a1    = ds_emos['a1']
    b0    = ds_emos['b0']
    b1    = ds_emos['b1']
    delta = ds_emos['delta']
 
    mu    = a0 + a1 * ens_mean
    sigma = np.abs(b0 + b1 * ens_std)
    sigma = np.maximum(sigma, EPS)
    mu    = np.maximum(mu, delta + 0.1)
 
    k, theta = csgd_params_from_moments(mu, sigma, delta)
 
    # P(Y = 0) = P(Gamma(k, theta) + delta <= 0) = F_gamma(-delta / theta; k)
    z_zero = np.maximum(-delta, 0.0) / np.maximum(theta, EPS)
    prob_zero = xr.apply_ufunc(gammainc, k, z_zero, dask='parallelized')
 
    # Predictive mean (censored): E[max(X + delta, 0)]
    # = (mu - delta) * (1 - F_gamma(-delta; k+1, theta)) + 0 * prob_zero
    # ≈ k * theta - delta * (1 - prob_zero)  for small prob_zero
    emos_mean = k * theta * (1.0 - xr.apply_ufunc(
        gammainc, k + 1.0, z_zero, dask='parallelized'
    ))

    delta_broadcast = delta.broadcast_like(k)
    
    return xr.Dataset({
        'emos_k':      k,
        'emos_theta':  theta,
        'emos_delta':  delta_broadcast,
        'emos_mu':     emos_mean,
        'emos_sigma':  sigma,
        'emos_prob0':  prob_zero,
    })

In [14]:
ds_calib = apply_emos_csgd(ds_input, ds_emos, var=VAR_CESM)
ds_calib = ds_calib.transpose('init_time', 'lead_time', 'lat', 'lon')
ds_calib = ds_calib.load()

In [22]:
ds_calib

<xarray.Dataset> Size: 9GB
Dimensions:     (lat: 192, lead_time: 3650, lon: 288, init_time: 1)
Coordinates:
  * lat         (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lead_time   (lead_time) int64 29kB 0 1 2 3 4 5 ... 3645 3646 3647 3648 3649
  * lon         (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
  * init_time   (init_time) int64 8B 2011
Data variables:
    emos_k      (init_time, lead_time, lat, lon) float64 2GB nan nan ... nan nan
    emos_theta  (init_time, lead_time, lat, lon) float64 2GB nan nan ... nan nan
    emos_delta  (init_time, lead_time, lat, lon) float32 807MB nan nan ... nan
    emos_mu     (init_time, lead_time, lat, lon) float64 2GB nan nan ... nan nan
    emos_sigma  (init_time, lead_time, lat, lon) float64 2GB nan nan ... nan nan
    emos_prob0  (init_time, lead_time, lat, lon) float64 2GB nan nan ... nan nan

In [15]:
def crps_csgd(mu, sigma, delta, obs):
    """
    Closed-form CRPS for the Censored Shifted Gamma Distribution.
 
    Parameters
    ----------
    mu, sigma, delta : array-like, CSGD location, scale, shift
    obs              : array-like, observed precipitation (>= 0)
 
    Returns
    -------
    crps : array, same shape as obs
 
    Based on Scheuerer & Hamill (2015) Eq. 3-5, using the identity
    for CRPS of a censored distribution at zero:
 
        CRPS = (y - delta) * (2*F_y - 1)
             - (mu - delta) * (2*F_0*F_y_k   - F_0^2 - 2*F_y_kp1 + 1)
             + (mu - delta) * B(0.5, k) / (k * pi)    (Gamma normalization term)
 
    where F_y = CDF at y, F_0 = CDF at 0, and F_y_k, F_y_kp1 are regularised
    incomplete gamma values at shape k and k+1.
    """
    k, theta = csgd_params_from_moments(mu, sigma, delta)
 
    # Standardise: z = (value - delta) / theta
    y = np.asarray(obs, dtype=np.float64)
    z_obs = np.maximum(y - delta, 0.0) / np.maximum(theta, EPS)
    z_zero = np.maximum(-delta, 0.0) / np.maximum(theta, EPS)
 
    # Regularised lower incomplete gamma: P(k, x) = gammainc(k, x)
    F_obs  = gammainc(k, z_obs)           # CDF at obs
    F_zero = gammainc(k, z_zero)          # CDF at 0  (= P(Y <= 0))
 
    # Terms for the CRPS formula
    # Term 1: observation part
    term1 = (y - delta) * (2.0 * F_obs - 1.0)
 
    # Clamp negative part (censoring at zero)
    term1 = term1 + delta * (2.0 * F_zero - 1.0)
    # Simpler: for censored, replace y with max(y, 0) in the formula
    # and add the censoring correction
 
    # Term 2: mean of the distribution * calibration factor
    # E[X] for Gamma(k, theta) = k * theta = (mu - delta)
    mu_shifted = k * theta
 
    # For two iid draws X1, X2 ~ Gamma(k, theta):
    # E|X1 - X2| = 2 * theta * k * Beta(k, 0.5) / sqrt(pi)
    # But with censoring we need the full expression.
 
    # Use the direct CRPS formula for censored Gamma from Scheuerer & Hamill:
    #   CRPS = y*(2*Fy - 1) - mu_s*(2*F0*Pk(z_y) - F0^2 + Pk+1(z_y)... )
    # where Pk means regularised incomplete gamma at shape k.
 
    # Simplified stable implementation via the Baran & Nemoda (2016) form:
    F_obs_kp1  = gammainc(k + 1.0, z_obs)
    F_zero_kp1 = gammainc(k + 1.0, z_zero)
 
    # Beta function term: B(0.5, k) = Gamma(0.5)*Gamma(k) / Gamma(k+0.5)
    log_B = (gammaln(0.5) + gammaln(k) - gammaln(k + 0.5))
    B_half_k = np.exp(log_B)
 
    crps = (
        np.maximum(y, 0.0) - np.maximum(y, 0.0) * 2.0 * F_obs
        + mu_shifted * (2.0 * F_obs_kp1 - 1.0)
        - mu_shifted * F_zero * (2.0 * F_zero_kp1 / np.maximum(F_zero, EPS) - F_zero)
        # ↑ careful: when F_zero ~ 0, this term vanishes anyway
    )
    # Correct version using the full closed form:
    crps = (
        np.maximum(y, 0.0) * (2.0 * F_obs - 1.0)
        - mu_shifted * (
            2.0 * F_obs_kp1
            - 2.0 * F_zero * F_zero_kp1
            + F_zero ** 2
            - B_half_k / np.sqrt(np.pi)
        )
    )
 
    # Censoring correction: add back the point mass contribution
    # For obs = 0 cases, the formula above already handles it through F_obs = F_zero
    # Final CRPS should be non-negative
    crps = np.maximum(crps, 0.0)
 
    return crps.astype(np.float32)
 
 
def mean_crps_csgd(params, ens_mean, ens_std, obs, min_mu_shift=0.1):
    """
    Objective function: mean CRPS over training samples.
 
    params: [a0, a1, b0, b1, delta]
    """
    a0, a1, b0, b1, delta = params
 
    mu    = a0 + a1 * ens_mean
    sigma = np.abs(b0 + b1 * ens_std)
    sigma = np.maximum(sigma, EPS)
 
    # Ensure mu > delta for valid Gamma params
    mu = np.maximum(mu, delta + min_mu_shift)
 
    try:
        c = crps_csgd(mu, sigma, delta, obs)
        result = np.nanmean(c)
        if np.isnan(result) or np.isinf(result):
            return 1e10
        return result
    except Exception:
        return 1e10

In [16]:
def crps_verif(ds_calib, ds_input, ds_target, date_idx,
                     var=VAR, init_idx=None):
    """
    Vectorized CRPS verification for one or more initializations.
    Compares raw ensemble Gaussian approximation vs CSGD-EMOS.
 
    Parameters
    ----------
    init_idx : int, list, or None (all)
 
    Returns
    -------
    raw_crps, emos_crps : arrays
        (lead_time, lat, lon) if single init, else (n_init, lead_time, lat, lon)
    """
    n_lead = ds_input.sizes['lead_time']
    n_lat  = ds_input.sizes['lat']
    n_lon  = ds_input.sizes['lon']
 
    target_vals = np.maximum(ds_target[var].values.astype(np.float32), 0.0)
 
    squeeze = False
    if init_idx is None:
        init_idx = np.arange(ds_input.sizes['init_time'])
    elif isinstance(init_idx, (int, np.integer)):
        init_idx = [init_idx]
        squeeze = True
    init_idx = np.asarray(init_idx)
    n_sel = len(init_idx)
 
    raw_crps  = np.full((n_sel, n_lead, n_lat, n_lon), np.nan, dtype=np.float32)
    emos_crps = np.full_like(raw_crps, np.nan)
 
    for j, ii in enumerate(init_idx):
 
        # Obs
        obs = np.full((n_lead, n_lat, n_lon), np.nan, dtype=np.float32)
        valid_leads = date_idx[ii] >= 0
        lead_idx = np.where(valid_leads)[0]
        time_idx = date_idx[ii, lead_idx]
        obs[lead_idx, :, :] = target_vals[time_idx, :, :]
 
        # Raw ensemble → Gaussian CRPS as baseline
        ens = ds_input[var].isel(init_time=ii)
        raw_mu  = np.maximum(ens.mean('member').values.astype(np.float32), 0.0)
        raw_std = np.maximum(ens.std('member', ddof=1).values.astype(np.float32), EPS)
 
        from scipy.stats import norm
        z = (np.maximum(obs, 0.0) - raw_mu) / raw_std
        raw_crps[j] = raw_std * (
            z * (2.0 * norm.cdf(z) - 1.0) + 2.0 * norm.pdf(z) - 1.0 / np.sqrt(np.pi)
        )
 
        # EMOS CSGD CRPS
        mu_e  = ds_calib['emos_mu'].isel(init_time=ii).values.astype(np.float32)
        sig_e = ds_calib['emos_sigma'].isel(init_time=ii).values.astype(np.float32)
        dlt_e = ds_calib['emos_delta'].isel(init_time=ii).values.astype(np.float32)
        sig_e = np.maximum(sig_e, EPS)
 
        emos_crps[j] = crps_csgd(mu_e, sig_e, dlt_e, np.maximum(obs, 0.0))
 
        # Mask missing
        invalid = np.isnan(obs)
        raw_crps[j][invalid]  = np.nan
        emos_crps[j][invalid] = np.nan
 
    # Summary
    raw_mean  = np.nanmean(raw_crps)
    emos_mean = np.nanmean(emos_crps)
    n_valid   = np.sum(~np.isnan(raw_crps))
    print(f"CRPS verification ({n_sel} init(s), {n_valid:,} valid cells):")
    print(f"  Raw (Gaussian) : {raw_mean:.4f}")
    print(f"  EMOS (CSGD)    : {emos_mean:.4f}")
    print(f"  Improvement    : {100 * (raw_mean - emos_mean) / raw_mean:.1f}%")
 
    if squeeze:
        raw_crps  = raw_crps[0]
        emos_crps = emos_crps[0]
 
    return raw_crps, emos_crps

In [17]:
raw_crps, emos_crps = crps_verif(ds_calib, ds_input, ds_target, date_idx, var=VAR_CESM)

/glade/derecho/scratch/ksha/tmp/ipykernel_39030/536235030.py:69: RuntimeWarning: Mean of empty slice
  emos_mean = np.nanmean(emos_crps)


CRPS verification (1 init(s), 201,830,400 valid cells):
  Raw (Gaussian) : 1.9536
  EMOS (CSGD)    : nan
  Improvement    : nan%


In [18]:
raw_crps.shape

(1, 3650, 192, 288)

In [19]:
emos_crps.shape

(1, 3650, 192, 288)

In [20]:
ds_verif = xr.Dataset(
    {
        "CRPS_CESM": (["init_time", "lead_time", "lat", "lon"], raw_crps),
        "CRPS_EMOS": (["init_time", "lead_time", "lat", "lon"], emos_crps),
    },
    coords={
        "init_time": verif_years,
        "lead_time": np.arange(3650),
        "lat": ds_input.lat.values,
        "lon": ds_input.lon.values,
    },
)

In [21]:
fn_save = f'/glade/derecho/scratch/ksha/EPRI_data/PP_verif/TP_EMOS_{verif_years[0]}_{verif_years[-1]}.zarr'
ds_verif.to_zarr(fn_save, mode='w')


KeyboardInterrupt



### Check

In [ ]:
for i in range(10):
    plt.figure()
    plt.plot(emos_crps[i, :, 99, 199])
    plt.plot(raw_crps[i, :, 99, 199])
    
    plt.xlabel('10 year daily prediction', fontsize=14)
    plt.ylabel('TP CRPS', fontsize=14)
    plt.title(f'{verif_years[i]} init', fontsize=14)